# 2. US: real-time implementation, post-publication sample, trading costs

The paper's scaling constant $c$ uses the full sample. Here $c_t$ is re-estimated each month
using only past data (expanding window, 120-month warm-up), with an optional cap on leverage.

In [1]:
import pandas as pd

from data_loaders import load_us_factors
from vol_tools import (realized_variance, vol_managed, vol_managed_rt,
                       spanning_regression, sharpe, net_of_cost, vol_buckets)

In [2]:
monthly_all, daily_all = load_us_factors()
monthly = monthly_all.loc[:"2015-12"]
daily = daily_all.loc[:"2015-12-31"]
print("full data ends:", monthly_all.index[-1])

factors = {"Mkt": "Mkt-RF", "Mom": "Mom"}
f     = {n: monthly[c].dropna() for n, c in factors.items()}
rv    = {n: realized_variance(daily[c].dropna()) for n, c in factors.items()}
f_all = {n: monthly_all[c].dropna() for n, c in factors.items()}
rv_all = {n: realized_variance(daily_all[c].dropna()) for n, c in factors.items()}

full data ends: 2026-09


## 2.1 Full-sample c vs. real-time c (paper sample, ends 2015)

All statistics are computed over the same months (after the warm-up period).
Alpha magnitudes depend on the average position size, so compare Sharpe ratios and t-statistics.

In [3]:
for name in factors:
    fs_full, _, _ = vol_managed(f[name], rv[name])
    print(f"===== {name} =====")
    for cap in [None, 2, 1.5]:
        fs_rt, w_rt = vol_managed_rt(f[name], rv[name], cap=cap)
        same = fs_rt.index
        r_full = spanning_regression(fs_full.loc[same], f[name])
        r_rt = spanning_regression(fs_rt, f[name])
        print(f"cap={cap}: Sharpe orig {sharpe(f[name].loc[same]):.2f} / "
              f"full-sample c {sharpe(fs_full.loc[same]):.2f} / real-time {sharpe(fs_rt):.2f} | "
              f"alpha t: full {r_full['alpha'] / r_full['alpha_se']:.2f}, "
              f"real-time {r_rt['alpha'] / r_rt['alpha_se']:.2f} | "
              f"avg position {w_rt.mean():.2f}, max {w_rt.max():.1f}")

===== Mkt =====
cap=None: Sharpe orig 0.48 / full-sample c 0.49 / real-time 0.45 | alpha t: full 1.93, real-time 1.45 | avg position 1.65, max 18.9
cap=2: Sharpe orig 0.48 / full-sample c 0.49 / real-time 0.48 | alpha t: full 1.93, real-time 1.31 | avg position 1.17, max 2.0
cap=1.5: Sharpe orig 0.48 / full-sample c 0.49 / real-time 0.50 | alpha t: full 1.93, real-time 1.37 | avg position 1.02, max 1.5
===== Mom =====
cap=None: Sharpe orig 0.56 / full-sample c 1.02 / real-time 0.76 | alpha t: full 7.03, real-time 4.35 | avg position 2.28, max 29.6
cap=2: Sharpe orig 0.56 / full-sample c 1.02 / real-time 0.92 | alpha t: full 7.03, real-time 5.07 | avg position 1.28, max 2.0
cap=1.5: Sharpe orig 0.56 / full-sample c 1.02 / real-time 0.87 | alpha t: full 7.03, real-time 4.27 | avg position 1.07, max 1.5


## 2.2 Post-publication sample (2016 onward)

The paper's data end in 2015 and it was published in 2017, so 2016 onward is a genuine out-of-sample period.
With only ~10 years of data, Sharpe ratio estimates have a standard error of roughly 0.3.

In [4]:
for name in factors:
    fs_rt, _ = vol_managed_rt(f_all[name], rv_all[name], cap=2)
    print(f"===== {name} (real-time, cap=2) =====")
    for label, start, end in [("before 2016", None, "2015-12"), ("2016 onward", "2016-01", None)]:
        y = fs_rt.loc[start:end]
        r = spanning_regression(y, f_all[name])
        print(f"{label}: N {len(y)} | Sharpe orig {sharpe(f_all[name].loc[y.index]):.2f} / "
              f"managed {sharpe(y):.2f} | alpha {r['alpha']:.2f} (t {r['alpha'] / r['alpha_se']:.2f})")

===== Mkt (real-time, cap=2) =====
before 2016: N 953 | Sharpe orig 0.48 / managed 0.48 | alpha 1.70 (t 1.31)
2016 onward: N 129 | Sharpe orig 0.83 / managed 0.70 | alpha 1.52 (t 0.51)
===== Mom (real-time, cap=2) =====
before 2016: N 948 | Sharpe orig 0.56 / managed 0.92 | alpha 6.69 (t 5.07)
2016 onward: N 129 | Sharpe orig 0.10 / managed 0.19 | alpha 0.57 (t 0.55)


## 2.3 Trading costs

Net return $= w_t f_t - \kappa\,|w_t - w_{t-1}|$. Only the cost of changing overall exposure is included;
the cost of rebalancing the factor portfolio itself (large for momentum) is not.

In [5]:
for name in factors:
    fs_rt, w_rt = vol_managed_rt(f_all[name], rv_all[name], cap=2)
    s0 = sharpe(f_all[name].loc[fs_rt.index])
    print(f"===== {name} ===== avg monthly turnover {w_rt.diff().abs().mean():.2f}, original Sharpe {s0:.2f}")
    for kappa in [0, 0.001, 0.005, 0.01]:
        print(f"  cost {kappa:.1%}: managed Sharpe {sharpe(net_of_cost(fs_rt, w_rt, kappa)):.2f}")

===== Mkt ===== avg monthly turnover 0.40, original Sharpe 0.52
  cost 0.0%: managed Sharpe 0.50
  cost 0.1%: managed Sharpe 0.47
  cost 0.5%: managed Sharpe 0.35
  cost 1.0%: managed Sharpe 0.21
===== Mom ===== avg monthly turnover 0.35, original Sharpe 0.51
  cost 0.0%: managed Sharpe 0.86
  cost 0.1%: managed Sharpe 0.83
  cost 0.5%: managed Sharpe 0.69
  cost 1.0%: managed Sharpe 0.52


## 2.4 The paper's premise: does volatility predict returns?

Months sorted into terciles by last month's realized variance (annualized).

In [6]:
for name in factors:
    print(f"US {name}")
    print(vol_buckets(f_all[name], rv_all[name]).round(3), "\n")

US Mkt
         mean    std  sharpe
bucket                      
low     0.086  0.112   0.765
mid     0.068  0.147   0.460
high    0.095  0.259   0.369 

US Mom
         mean    std  sharpe
bucket                      
low     0.107  0.074   1.440
mid     0.089  0.120   0.740
high    0.027  0.244   0.112 

